# Chapter 3 — ProofFrame Recheck and Rule Overlays

Four capabilities shipped during the Round Story routemap (Batches 4 and 5a/b/c):

1. **ProofFrame Rechecker** (Batch 4, `recheck_proof_frame`) — given a `SupportArtifact` and an `EvaluationOverlay`, recheck per-atom verdicts and return a 3-status frame verdict (`still_valid` / `invalidated` / `unknown`).
2. **Rule Disable** (Batch 5a, `check_rule_disable_action`) — temporarily drop one body atom from a rule and observe the variant rows it would now produce.
3. **Rule Literal Replace** (Batch 5b, `check_rule_literal_replace_action`) — swap a constant literal in a rule body atom (e.g. region 'us' → 'eu').
4. **Rule Add Condition** (Batch 5c, `check_rule_add_condition_action`) — append one filter atom (no new variable binding) to a rule body branch.

All three rule overlays are single-action MVPs that return both a `variant_rows` projection and a `proof_frame: ProofFrameRecheckResult` describing how the original frame was affected.

**Series navigation**

- Previous: `02_overlay_why_not_frontier.ipynb`.
- Next: `04_round_persistence_diff.ipynb` (Batch 6 round events + Batch 7 ProofFrame diff).
- Integrated walkthrough: `round_story_full_demo.py`.

## Setup

ProofFrame and rule-overlay calls need extra context beyond the bare fixture: a `SupportArtifact` (from a prior Check), a fact overlay (to drive the ProofFrame recheck), and a `RuleContext` (rule_spec + a synthetic support artifact for the rule overlays).

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'examples').exists() and (_repo_root.parent / 'examples').exists():
    _repo_root = _repo_root.parent
for sub in ('src', 'examples'):
    candidate = _repo_root / sub
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import round_story_full_demo as demo  # noqa: E402

fixture = demo._build_fixture()
alice = fixture.people['alice']
rule_context = demo.RuleContext(
    rule_spec=demo._rule_spec(fixture.index),
    support_artifact=demo._rule_support_artifact(alice),
)
# Warm-up Check + Fact Overlay so we have the SupportArtifact and EvaluationOverlay
# the ProofFrame recheck consumes.
_check_request, _check_result, support = demo._phase_check(fixture, verbose=False)
_fo_request, _fo_result, fact_overlay = demo._phase_fact_overlay(fixture, verbose=False)
print('Setup complete: fixture, rule_context, support, fact_overlay all ready.')

## 1. ProofFrame Rechecker (Batch 4)

`recheck_proof_frame(...)` answers: *given the per-atom proof Alice's success rested on, does that proof still hold under this overlay?*

The phase below runs two rechecks against the same support artifact:

- a **baseline** with an empty `EvaluationOverlay` — the proof must be `still_valid`.
- an **overlay** that flips Alice's age from 25 to 30 — the original age witness is invalidated, so the frame status drops to `invalidated`.

The result carries per-atom verdicts (`atom_verdicts`) plus the aggregate frame status.

In [ ]:
(
    baseline_pf_request,
    baseline_pf_result,
    overlay_pf_request,
    overlay_pf_result,
) = demo._phase_proofframe(fixture, support, fact_overlay, verbose=True)

print(f'\nbaseline status = {baseline_pf_result.status}')
print(f'overlay status  = {overlay_pf_result.status}')
for verdict in overlay_pf_result.atom_verdicts:
    print(
        f'  atom {verdict.atom_key}: verdict={verdict.verdict} '
        f'affected_action_indices={verdict.affected_action_indices}'
    )

## 2. Rule Disable (Batch 5a)

`check_rule_disable_action(...)` evaluates a rule body with one designated atom dropped. The original rule is `name(?p) ∧ age(?p, 25) ∧ region(?p, 'us')`; disabling atom 3 (the region equality) widens the matching universe.

The result reports `variant_rows` (the new bindings the disabled rule would produce) plus a nested ProofFrame indicating whether Alice's original frame is still valid (it is invalidated because the original rule no longer matches the same row set).

In [ ]:
rule_disable = demo._phase_rule_disable(fixture, rule_context, verbose=True)

print(f'\nstatus              = {rule_disable.status}')
print(f'variant_rows count  = {len(rule_disable.variant_rows)}')
for row in rule_disable.variant_rows:
    print(f'  variant row: {row}')
print(f'proof_frame status  = {rule_disable.proof_frame.status}')

## 3. Rule Literal Replace (Batch 5b)

`check_rule_literal_replace_action(...)` swaps a constant literal inside a body atom. The phase below replaces the rhs of the region equality from `'us'` to `'eu'`, narrowing the variant row to Bob (the only seeded EU person).

In [ ]:
rule_literal_replace = demo._phase_rule_literal_replace(fixture, rule_context, verbose=True)

print(f'\nstatus              = {rule_literal_replace.status}')
print(f'variant_rows        = {rule_literal_replace.variant_rows}')
print(f'proof_frame status  = {rule_literal_replace.proof_frame.status}')

## 4. Rule Add Condition (Batch 5c)

`check_rule_add_condition_action(...)` inserts one extra filter atom into a rule body branch. The added atom in this phase is `lt($age, 20)` — narrowing the result to Dave (the only seeded person under 20).

Batch 5c ships the *filter-only* slice; introducing new variable binders is deferred per the routemap closure decision.

In [ ]:
rule_add_condition = demo._phase_rule_add_condition(fixture, rule_context, verbose=True)

print(f'\nstatus              = {rule_add_condition.status}')
print(f'variant_rows        = {rule_add_condition.variant_rows}')
print(f'proof_frame status  = {rule_add_condition.proof_frame.status}')

## 5. Aggregate verification

`run_proofframe_rule_overlay_demo(...)` re-runs the four phases against a fresh fixture and returns the smoke-test status dict the unittest asserts on.

In [ ]:
summary = demo.run_proofframe_rule_overlay_demo(verbose=False)
expected = {
    'proofframe': 'invalidated',
    'rule_disable': 'completed',
    'rule_literal_replace': 'completed',
    'rule_add_condition': 'completed',
}

print(f'Chapter summary : {summary}')
print(f'Expected        : {expected}')
assert summary == expected, summary
print('\n✓ Chapter 3 aggregate matches expected smoke contract.')

## Where to next

- **Chapter 4 (`04_round_persistence_diff.ipynb`)** — Batch 6 durable round events and Batch 7 ProofFrame diff over two finalized rounds.
- **Module reference** — `src/kernel/application/docs/01_overview.md` for ProofFrame + rule-overlay protocol DTOs; archived blueprints under `docs/blueprints/archive/2026-05-05_*` for Batch 4 / 5a / 5b / 5c design rationale.
- **Boundary** — All four capabilities are advanced-importable (`kernel.application.{proofframe_runtime, rule_disable_runtime, rule_literal_replace_runtime, rule_add_condition_runtime}`); v0.1 ships no SDK shells per Batch 8.